# 05 — EDA VitalDB cho UC04

Notebook tương tác dùng cùng module với báo cáo tự động (`safeanes.eda`, `safeanes.eda_plots`).
Báo cáo đầy đủ: `reports/EDA/REPORT.md` (sinh bởi `python scripts/eda/run_eda.py`).

**Phạm vi:** mô tả cohort/track dùng toàn bộ ca; mọi phân tích outcome chỉ dùng ca ngoài global test (`unseen_test`).
Dữ liệu cần có: `data/vitaldb_full/{meta,raw,cases}`, `data/sequences_full`, `data/beats_full`
(tạo bằng `scripts/data/fetch_uc04_tracks.py` và `scripts/data/extract_beats.py`).

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "src/safeanes").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT / "src", ROOT / ".local_deps"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
import numpy as np, pandas as pd
import matplotlib
from safeanes import eda, eda_plots as plots
CACHE = ROOT / "data/eda_cache"
FIG = ROOT / "reports/EDA/figures"
pd.set_option("display.max_columns", 50)

## 1. Cohort

In [ ]:
manifest = eda.load_manifest(ROOT)
display(eda.cohort_flow(manifest))
display(eda.demographics(manifest))
display(eda.category_table(manifest, "department"))

## 2. Track liên quan UC04

In [ ]:
avail = eda.track_availability(ROOT, manifest)
display(avail.sort_values("% eligible", ascending=False))

## 3. Phân tích theo ca (biến cố, quỹ đạo, thay đổi trước IOH)

Dùng cache của `run_eda.py` nếu có; nếu không, chạy thử trên `LIMIT` ca.

In [ ]:
LIMIT = 200
if (CACHE / "events.parquet").exists():
    cases = pd.read_parquet(CACHE / "cases.parquet")
    events = pd.read_parquet(CACHE / "events.parquet")
    traj = np.load(CACHE / "trajectories.npy")
else:
    cases, events, traj = eda.analyze_cohort(ROOT, manifest, workers=4, limit=LIMIT)
ev = events[events.kind.eq("event")]
print(len(cases), "ca;", len(ev), "biến cố;", round(len(ev) / cases.monitored_h.sum(), 3), "biến cố/giờ")
ev[["duration_s", "min_map", "auc65_mmHg_min", "min_after_opstart"]].describe()

In [ ]:
from IPython.display import Image
plots.events_overview(events, cases, FIG / "events_overview.png")
Image(FIG / "events_overview.png")

In [ ]:
plots.trajectories(events, traj, FIG / "trajectories_core.png",
                   ["map", "hr", "bt_sv_lz", "bt_svr_lz", "bt_co_lz", "bt_ppv", "bt_dpdt", "bt_pp"])
Image(FIG / "trajectories_core.png")

In [ ]:
plots.trajectories(events, traj, FIG / "trajectories_context.png",
                   ["bis", "ppf_ce", "rftn_ce", "mac", "cvp", "peep", "ev_sv", "ev_svr", "ev_svv", "vg_svv", "phen_rate", "nibp_mbp"])
Image(FIG / "trajectories_context.png")

## 4. Phân rã SV × HR × SVR và mẫu hình heuristic (mô tả, không phải nhãn)

In [ ]:
dec = eda.decompose(events)
display(dec.groupby("kind").pattern.value_counts(normalize=True).unstack(0).round(3))
plots.decomposition(dec, FIG / "decomposition.png")
Image(FIG / "decomposition.png")

In [ ]:
centroids, clustered = eda.cluster_changes(dec)
display(centroids)

## 5. Proxy vs EV1000, NIBP vs ART, mất máu

In [ ]:
proxy = eda.proxy_vs_ev1000(ROOT, manifest, limit=100)
display(proxy.describe().T)

In [ ]:
pairs, nibp = eda.nibp_vs_art(ROOT, manifest, max_cases=100)
nibp

In [ ]:
bleed = eda.labs_bleeding(ROOT, manifest, cases)
bleed[["events_per_h", "hb_drop", "intraop_ebl", "intraop_rbc", "duration_h"]].corr("spearman").round(3)

## 6. Phân nhóm

In [ ]:
for col, table in eda.subgroup_rates(manifest, cases).items():
    print(col); display(table)

## 7. Insight: mô tả, nhiễu, quan hệ, feature ảnh hưởng

Cùng module với `scripts/eda/run_insights.py` → `reports/EDA/INSIGHTS.md`. Dùng cache `data/eda_cache/insight_table.parquet` nếu có
(bảng decision row 60 s × 173 feature của các ca phát triển); nếu không, dựng thử trên `LIMIT` ca.

In [ ]:
from safeanes import insights as ins
if (CACHE / "insight_table.parquet").exists():
    table = pd.read_parquet(CACHE / "insight_table.parquet")
else:
    table = ins.build_table(ROOT, manifest, workers=4, limit=LIMIT)
dictionary = ins.dictionary(table)
display(dictionary.groupby("nhóm").agg(so_feature=("feature", "size"), thieu_median=("% thiếu", "median")))
display(dictionary[dictionary.feature.isin(ins.KEY_FEATURES)].sort_values("% thiếu", ascending=False).head(15))

In [ ]:
noise_path = ROOT / "reports/EDA/insight_noise.csv"
noise = pd.read_csv(noise_path) if noise_path.exists() else ins.noise_profile(ROOT, manifest, workers=4, limit=LIMIT)[1]
display(noise)

In [ ]:
corr = ins.spearman(table)
plots.corr_heatmap(corr, FIG / "insight_corr.png")
Image(FIG / "insight_corr.png")

In [ ]:
future = ins.future_association(table)
display(future[~future["nhóm"].isin(["MAP", "SBP/DBP"])].head(12))
risk = ins.level_trend_map(table)
plots.level_trend(risk, FIG / "insight_level_trend.png")
Image(FIG / "insight_level_trend.png")

In [ ]:
uni = ins.univariate_auroc(table)
display(uni.groupby("nhóm").head(1))
imp, perf = ins.shap_importance(table, "y_300")
print(perf)
display(imp.head(20))
display(imp.groupby("nhóm")["share_%"].sum().sort_values(ascending=False))